## Introduction

In this lab assessment, you'll practice your knowledge of JOIN statements and subqueries, using various types of joins and various methods for specifying the links between them.


## Learning Objectives

- Write SQL queries that make use of various types of joins
- Choose and perform whichever type of join is best for retrieving desired data
- Write subqueries to decompose complex queries


In [1]:
# CodeGrade step0
# Run this cell without changes

import sqlite3
import pandas as pd

conn = sqlite3.connect('data.sqlite')
pd.read_sql("""SELECT * FROM sqlite_master""", conn)

,type,name,tbl_name,rootpage,sql
0,table,orderdetails,orderdetails,2,"CREATE TABLE `orderdetails` (`orderNumber`, `p..."
1,table,payments,payments,28,"CREATE TABLE `payments` (`customerNumber`, `ch..."
2,table,offices,offices,32,"CREATE TABLE `offices` (`officeCode`, `city`, ..."
3,table,customers,customers,33,"CREATE TABLE `customers` (`customerNumber`, `c..."
4,table,orders,orders,38,"CREATE TABLE `orders` (`orderNumber`, `orderDa..."
5,table,productlines,productlines,46,"CREATE TABLE `productlines` (`productLine`, `t..."
6,table,products,products,47,"CREATE TABLE `products` (`productCode`, `produ..."
7,table,employees,employees,56,"CREATE TABLE `employees` (`employeeNumber`, `l..."


## Part 1: Join and Filter


### Step 1

Return the first and last names and the job titles for all employees in Boston.


In [2]:
# CodeGrade step1
df_boston = pd.read_sql("""
    SELECT e.firstName, e.lastName, e.jobTitle
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode
    WHERE o.city = 'Boston'
""", conn)

df_boston

,firstName,lastName,jobTitle
0,Julie,Firrelli,Sales Rep
1,Steve,Patterson,Sales Rep


### Step 2

Are there any offices that have zero employees?


In [3]:
# CodeGrade step2
# A LEFT JOIN keeps all offices; NULL on the employee side means no employees
df_zero_emp = pd.read_sql("""
    SELECT o.officeCode, o.city
    FROM offices o
    LEFT JOIN employees e ON o.officeCode = e.officeCode
    WHERE e.employeeNumber IS NULL
""", conn)

df_zero_emp
# Empty DataFrame means every office has at least one employee — no 'ghost' locations

,officeCode,city


## Part 2: Type of Join


### Step 3

Return all employees' first and last name along with the city and state of their office. Include ALL employees and order by first name then last name.


In [4]:
# CodeGrade step3
# LEFT JOIN so employees without an office are still included
df_employee = pd.read_sql("""
    SELECT e.firstName, e.lastName, o.city, o.state
    FROM employees e
    LEFT JOIN offices o ON e.officeCode = o.officeCode
    ORDER BY e.firstName, e.lastName
""", conn)

df_employee

,firstName,lastName,city,state
0,Andy,Fixter,Sydney,
1,Anthony,Bow,San Francisco,CA
2,Barry,Jones,London,
3,Diane,Murphy,San Francisco,CA
4,Foon Yue,Tseng,NYC,NY
5,George,Vanauf,NYC,NY
6,Gerard,Bondur,Paris,
7,Gerard,Hernandez,Paris,
8,Jeff,Firrelli,San Francisco,CA
9,Julie,Firrelli,Boston,MA


### Step 4

Return contact info and sales rep employee number for any customer who has NOT placed an order. Sort alphabetically by contact last name.


In [5]:
# CodeGrade step4
# LEFT JOIN customers -> orders; NULL orderNumber means no order was placed
df_contacts = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName,
           c.phone, c.salesRepEmployeeNumber
    FROM customers c
    LEFT JOIN orders o ON c.customerNumber = o.customerNumber
    WHERE o.orderNumber IS NULL
    ORDER BY c.contactLastName
""", conn)

df_contacts

,contactFirstName,contactLastName,phone,salesRepEmployeeNumber
0,Raanan,"Altagar,G M",+ 972 9 959 8555,
1,Mel,Andersen,030-0074555,
2,Carmen,Anton,+34 913 728555,
3,Alejandra,Camino,(91) 745 6555,
4,Philip,Cramer,0555-09555,
5,Alexander,Feuer,0342-555176,
6,Keith,Franco,2035557845,1286
7,Peter,Franken,089-0877555,
8,Ed,Harrison,+41 26 425 50 01,
9,Karin,Josephs,0251-555259,


## Part 3: Built-in Function


### Step 5

Return all customer contacts with their payment amounts and dates, sorted by payment amount descending.


In [6]:
# CodeGrade step5
# CAST amount to REAL to ensure correct numeric sort (stored as TEXT in this DB)
df_payment = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName,
           p.paymentDate,
           CAST(p.amount AS REAL) AS amount
    FROM customers c
    JOIN payments p ON c.customerNumber = p.customerNumber
    ORDER BY CAST(p.amount AS REAL) DESC
""", conn)

df_payment

,contactFirstName,contactLastName,paymentDate,amount
0,Diego,Freyre,2005-03-18,120166.58
1,Diego,Freyre,2004-12-31,116208.40
2,Susan,Nelson,2003-08-15,111654.40
3,Eric,Natividad,2003-12-26,105743.00
4,Susan,Nelson,2005-03-05,101244.59
...,...,...,...,...
268,Carine,Schmitt,2004-12-18,1676.14
269,Pascale,Cartrain,2003-04-19,1627.56
270,Jonas,Bergulfsen,2003-10-28,1491.38
271,Pascale,Cartrain,2003-08-22,1128.20


## Part 4: Joining and Grouping


### Step 6

Return the employee number, first name, last name, and number of customers for employees whose customers have an average credit limit over 90k. Sort by number of customers (high to low).


In [7]:
# CodeGrade step6
df_credit = pd.read_sql("""
    SELECT e.employeeNumber, e.firstName, e.lastName,
           COUNT(c.customerNumber) AS num_customers
    FROM employees e
    JOIN customers c ON e.employeeNumber = c.salesRepEmployeeNumber
    GROUP BY e.employeeNumber
    HAVING AVG(CAST(c.creditLimit AS REAL)) > 90000
    ORDER BY num_customers DESC
""", conn)

df_credit

,employeeNumber,firstName,lastName,num_customers
0,1501,Larry,Bott,8
1,1370,Gerard,Hernandez,7
2,1165,Leslie,Jennings,6
3,1612,Peter,Marsh,5


### Step 7

Return product name, count of orders as 'numorders', and total quantity sold as 'totalunits'. Sort by totalunits highest to lowest.


In [8]:
# CodeGrade step7
df_product_sold = pd.read_sql("""
    SELECT p.productName,
           COUNT(od.orderNumber)     AS numorders,
           SUM(od.quantityOrdered)   AS totalunits
    FROM products p
    JOIN orderdetails od ON p.productCode = od.productCode
    GROUP BY p.productCode
    ORDER BY totalunits DESC
""", conn)

df_product_sold

,productName,numorders,totalunits
0,1992 Ferrari 360 Spider red,53,1808
1,1937 Lincoln Berline,28,1111
2,American Airlines: MD-11S,28,1085
3,1941 Chevrolet Special Deluxe Cabriolet,28,1076
4,1930 Buick Marquette Phaeton,28,1074
...,...,...,...
104,1999 Indy 500 Monte Carlo SS,25,855
105,1911 Ford Town Car,25,832
106,1936 Mercedes Benz 500k Roadster,25,824
107,1970 Chevy Chevelle SS 454,25,803


## Part 5: Multiple Joins


### Step 8

Return product name, product code, and total distinct customers who ordered each product as 'numpurchasers'. Sort by highest purchasers.


In [9]:
# CodeGrade step8
# 3-table join: products -> orderdetails -> orders (which has customerNumber)
df_total_customers = pd.read_sql("""
    SELECT p.productName,
           p.productCode,
           COUNT(DISTINCT o.customerNumber) AS numpurchasers
    FROM products p
    JOIN orderdetails od ON p.productCode = od.productCode
    JOIN orders o        ON od.orderNumber = o.orderNumber
    GROUP BY p.productCode
    ORDER BY numpurchasers DESC
""", conn)

df_total_customers

,productName,productCode,numpurchasers
0,1992 Ferrari 360 Spider red,S18_3232,40
1,Boeing X-32A JSF,S72_1253,27
2,1934 Ford V8 Coupe,S18_2957,27
3,1972 Alfa Romeo GTA,S10_4757,27
4,1952 Alpine Renault 1300,S10_1949,27
...,...,...,...
104,1958 Chevy Corvette Limited Edition,S24_2840,19
105,2002 Chevy Corvette,S24_3432,18
106,1969 Chevrolet Camaro Z28,S24_3191,18
107,1952 Citroen-15CV,S24_2887,18


### Step 9

Return the count of customers per office as 'n_customers', along with office code and city.


In [10]:
# CodeGrade step9
# Chain: offices -> employees -> customers (via salesRepEmployeeNumber)
df_customers = pd.read_sql("""
    SELECT o.officeCode,
           o.city,
           COUNT(DISTINCT c.customerNumber) AS n_customers
    FROM offices o
    JOIN employees e  ON o.officeCode      = e.officeCode
    JOIN customers c  ON e.employeeNumber  = c.salesRepEmployeeNumber
    GROUP BY o.officeCode
""", conn)

df_customers

,officeCode,city,n_customers
0,1,San Francisco,12
1,2,Boston,12
2,3,NYC,15
3,4,Paris,29
4,5,Tokyo,5
5,6,Sydney,10
6,7,London,17


## Part 6: Subquery


### Step 10

Return employee number, first name, last name, office city and office code for employees who sold products ordered by fewer than 20 customers.


In [11]:
# CodeGrade step10
# Subquery first identifies product codes with < 20 distinct purchasers,
# then the outer query finds the employees who sold those products.
df_under_20 = pd.read_sql("""
    SELECT DISTINCT e.employeeNumber, e.firstName, e.lastName,
                    o.city, o.officeCode
    FROM employees e
    JOIN offices    o   ON e.officeCode      = o.officeCode
    JOIN customers  c   ON e.employeeNumber  = c.salesRepEmployeeNumber
    JOIN orders     ord ON c.customerNumber  = ord.customerNumber
    JOIN orderdetails od ON ord.orderNumber  = od.orderNumber
    WHERE od.productCode IN (
        -- Subquery: products purchased by fewer than 20 distinct customers
        SELECT p.productCode
        FROM products p
        JOIN orderdetails od2 ON p.productCode   = od2.productCode
        JOIN orders       o2  ON od2.orderNumber = o2.orderNumber
        GROUP BY p.productCode
        HAVING COUNT(DISTINCT o2.customerNumber) < 20
    )
""", conn)

df_under_20

,employeeNumber,firstName,lastName,city,officeCode
0,1370,Gerard,Hernandez,Paris,4
1,1501,Larry,Bott,London,7
2,1337,Loui,Bondur,Paris,4
3,1166,Leslie,Thompson,San Francisco,1
4,1286,Foon Yue,Tseng,NYC,3
5,1612,Peter,Marsh,Sydney,6
6,1611,Andy,Fixter,Sydney,6
7,1401,Pamela,Castillo,Paris,4
8,1621,Mami,Nishi,Tokyo,5
9,1323,George,Vanauf,NYC,3


### Close the connection


In [12]:
# Run this cell without changes
conn.close()